# Tamil Speaker Attribute Classification

Tamil speech features using MFCCs, chromagrams and mel spectra, with classifier comparisons.

Stage: project experiment.


NAME: - SISIR KAUSTHUBH


ROLL NUMBER :- STUDENT_ID

Audio Classification using multiple features on Common-Voice Corpus

Task: Age Prediction


In [ ]:
!pip install datasets

In [ ]:
import librosa as lb
import numpy as np
import pandas as pd
from datasets import load_dataset
import timeit
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import librosa.display as display
import IPython.display as ipd
import time

Preparation of the data:

In [ ]:
dataset = load_dataset("mozilla-foundation/common_voice_11_0","ta")

In [ ]:
train = dataset["train"]
test = dataset["test"]

In [ ]:
dataset

In [ ]:
train_files, train_labels = dataset['train']["path"],dataset['train']["age"]
train_df = pd.DataFrame.from_dict({"path": train_files, "label": train_labels})
train_df = train_df[train_df["label"]!=""]
train_df = train_df[train_df["label"]!="others"]
train_files, train_labels = list(train_df["path"]), list([train_df["label"]])


test_files, test_labels = dataset['test']["path"],dataset['test']["age"]
test_df = pd.DataFrame.from_dict({"path": test_files, "label": test_labels})
test_df = test_df[test_df["label"]!=""]
test_df = test_df[test_df["label"]!="others"]
test_files, test_labels = list(test_df["path"]), list([test_df["label"]])

In [ ]:
test_df_balanced = pd.concat([test_df[test_df["label"] == "thirties"].head(300),test_df[test_df["label"] == "twenties"].head(500),test_df[test_df["label"] == "fourties"].head(500),test_df[test_df["label"] == "fifties"].head(500)])
test_files = list(test_df_balanced["path"])
test_labels = list(test_df_balanced["label"])

In [ ]:
train_df_balanced = pd.concat([train_df[train_df["label"] == "thirties"].head(500),train_df[train_df["label"] == "twenties"].head(500),train_df[train_df["label"] == "fourties"].head(500),train_df[train_df["label"] == "fifties"].head(500)])
train_files = list(train_df_balanced["path"])
train_labels = list(train_df_balanced["label"])

In [ ]:
print("number of train samples per class 'twenties':",train_labels.count("twenties"))
print("number of train samples per class 'thirties':",train_labels.count("thirties"))
print("number of train samples per class 'fourties':",train_labels.count("fourties"))
print("number of train samples per class 'fifties':",train_labels.count("fifties"))

In [ ]:
print("number of test samples per class 'twenties':",test_labels.count("twenties"))
print("number of test samples per class 'thirties':",test_labels.count("thirties"))
print("number of test samples per class 'fourties':",test_labels.count("fourties"))
print("number of test samples per class 'fifties':",test_labels.count("fifties"))

Data Observation and Feature Engineering

In [ ]:

twenties_file = train_files[train_labels.index("twenties")]
thirties_file = train_files[train_labels.index("thirties")]
fourties_file = train_files[train_labels.index("fourties")]
fifties_file = train_files[train_labels.index("fifties")]


x_twenties, sr = lb.load(twenties_file,sr=16000)
print(type(x_twenties),type(sr))
x_thirties, sr = lb.load(thirties_file,sr=16000)
print(type(x_thirties),type(sr))
x_fourties, sr = lb.load(fourties_file,sr=16000)
print(type(x_fourties),type(sr))
x_fifties, sr = lb.load(fifties_file,sr=16000)
print(type(x_fifties),type(sr))

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from IPython.core.display import HTML
HTML("""
<style>
  .output_png{
    display: table-cell;
    text-align: center;
    vertical-align: middle;
  }
""")

In [ ]:
plt.figure(figsize=(15,8))
plt.subplot(4,1,1)
plt.title('twenties')
display.waveshow(x_twenties,sr=sr)
plt.subplot(4,1,2)
display.waveshow(x_thirties,sr=sr)
plt.title('thirties')
plt.subplot(4,1,3)
display.waveshow(x_fourties,sr=sr)
plt.title('fourties')
plt.subplot(4,1,4)
display.waveshow(x_fifties,sr=sr)
plt.title('fifties')

In [ ]:


melspectrogram = lb.feature.melspectrogram(y=x_twenties, sr=sr, n_mels = 128, n_fft=512, hop_length = 200, fmax=8000, win_length=400,window='hann')
plt.figure(figsize=(10, 5))
lb.display.specshow(lb.power_to_db(S=melspectrogram, ref=np.mean),y_axis='mel',fmax=8000, x_axis='time',norm=Normalize(vmin=-20,vmax=20))
plt.colorbar(format='%+2.0f dB',label='Amplitude')
plt.ylabel('Mels')
plt.title('twenties Mel spectrogram')
plt.tight_layout()

melspectrogram = lb.feature.melspectrogram(y= x_thirties, sr=sr, n_mels = 128, n_fft=512, hop_length = 200, fmax=8000, win_length=400,window='hann')
plt.figure(figsize=(10, 5))
lb.display.specshow(lb.power_to_db(S=melspectrogram, ref=np.mean),y_axis='mel',fmax=8000, x_axis='time',norm=Normalize(vmin=-20,vmax=20))
plt.colorbar(format='%+2.0f dB',label='Amplitude')
plt.ylabel('Mels')
plt.title('thirties Mel spectrogram')
plt.tight_layout()

melspectrogram = lb.feature.melspectrogram(y=x_fourties, sr=sr, n_mels = 128, n_fft=512, hop_length = 200, fmax=8000, win_length=400,window='hann')
plt.figure(figsize=(10, 5))
lb.display.specshow(lb.power_to_db(S=melspectrogram, ref=np.mean),y_axis='mel',fmax=8000, x_axis='time',norm=Normalize(vmin=-20,vmax=20))
plt.colorbar(format='%+2.0f dB',label='Amplitude')
plt.ylabel('Mels')
plt.title('fourties Mel spectrogram')
plt.tight_layout()

melspectrogram = lb.feature.melspectrogram(y=x_fifties, sr=sr, n_mels = 128, n_fft=512, hop_length = 200, fmax=8000, win_length=400,window='hann')
plt.figure(figsize=(10, 5))
lb.display.specshow(lb.power_to_db(S=melspectrogram, ref=np.mean),y_axis='mel',fmax=8000, x_axis='time',norm=Normalize(vmin=-20,vmax=20))
plt.colorbar(format='%+2.0f dB',label='Amplitude')
plt.ylabel('Mels')
plt.title('fifties Mel spectrogram')
plt.tight_layout()



In [ ]:


mfc_coefficients = lb.feature.mfcc(y=x_twenties, sr=sr, n_fft=512, hop_length = 200, win_length=400,window='hann',n_mfcc=40)
plt.figure(figsize=(10, 5))
lb.display.specshow(mfc_coefficients, x_axis='time',norm=Normalize(vmin=-30,vmax=30))
plt.colorbar()
plt.yticks(())
plt.ylabel('MFC Coefficients')
plt.title('twenties MFC Coefficients')
plt.tight_layout()

mfc_coefficients = lb.feature.mfcc(y=x_thirties, sr=sr, n_fft=512, hop_length = 200, win_length=400,window='hann',n_mfcc=40)
plt.figure(figsize=(10, 5))
lb.display.specshow(mfc_coefficients, x_axis='time',norm=Normalize(vmin=-30,vmax=30))
plt.colorbar()
plt.yticks(())
plt.ylabel('MFC Coefficients')
plt.title('MFC Coefficients')
plt.title('thirties MFC Coefficients')
plt.tight_layout()

mfc_coefficients = lb.feature.mfcc(y=x_fourties, sr=sr, n_fft=512, hop_length = 200, win_length=400,window='hann',n_mfcc=40)
plt.figure(figsize=(10, 5))
lb.display.specshow(mfc_coefficients, x_axis='time',norm=Normalize(vmin=-30,vmax=30))
plt.colorbar()
plt.yticks(())
plt.ylabel('MFC Coefficients')
plt.title('fourties MFC Coefficients')
plt.tight_layout()

mfc_coefficients = lb.feature.mfcc(y=x_fifties, sr=sr, n_fft=512, hop_length = 200, win_length=400,window='hann',n_mfcc=40)
plt.figure(figsize=(10, 5))
lb.display.specshow(mfc_coefficients, x_axis='time',norm=Normalize(vmin=-30,vmax=30))
plt.colorbar()
plt.yticks(())
plt.ylabel('MFC Coefficients')
plt.title('fifties MFC Coefficients')
plt.tight_layout()

In [ ]:
chromagram = lb.feature.chroma_stft(y=x_twenties,sr=sr,n_fft = 512,hop_length=200,win_length=400)
plt.figure(figsize=(10, 4))
lb.display.specshow(chromagram,y_axis='chroma', x_axis='time')
plt.colorbar(label='Relative Intensity')
plt.title('twenties Chromagram')
plt.tight_layout()

chromagram = lb.feature.chroma_stft(y=x_thirties,sr=sr,n_fft = 512,hop_length=200,win_length=400)
plt.figure(figsize=(10, 4))
lb.display.specshow(chromagram,y_axis='chroma', x_axis='time')
plt.colorbar(label='Relative Intensity')
plt.title('thirties Chromagram')
plt.tight_layout()

chromagram = lb.feature.chroma_stft(y=x_fourties,sr=sr,n_fft = 512,hop_length=200,win_length=400)
plt.figure(figsize=(10, 4))
lb.display.specshow(chromagram,y_axis='chroma', x_axis='time')
plt.colorbar(label='Relative Intensity')
plt.title('fourties Chromagram')
plt.tight_layout()

chromagram = lb.feature.chroma_stft(y=x_fifties,sr=sr,n_fft = 512,hop_length=200,win_length=400)
plt.figure(figsize=(10, 4))
lb.display.specshow(chromagram,y_axis='chroma', x_axis='time')
plt.colorbar(label='Relative Intensity')
plt.title('fifties Chromagram')
plt.tight_layout()

Feature Extraction

In [ ]:
def feature_chromagram(waveform, sample_rate):
    # Compute Short-Time Fourier Transform (STFT)
    stft_spectrogram = np.abs(lb.stft(waveform, n_fft=512, hop_length=256, win_length=400, window='hann'))

    # Produce the chromagram for all STFT frames and get the mean of each column
    chromagram = np.mean(lb.feature.chroma_stft(S=stft_spectrogram, sr=sample_rate).T, axis=0)

    return chromagram

In [ ]:
def feature_melspectrogram(waveform, sample_rate):
    # Produce the mel spectrogram for all STFT frames and get the mean of each column
    melspectrogram = np.mean(lb.feature.melspectrogram(y=waveform, sr=sample_rate, n_mels=128, fmax=8000, n_fft=512, hop_length=256, win_length=400, window='hann').T, axis=0)

    return melspectrogram

In [ ]:
def feature_mfcc(waveform, sample_rate):
    # Compute the MFCCs for all STFT frames and get the mean of each column
    mfc_coefficients = np.mean(lb.feature.mfcc(y=waveform, sr=sample_rate, n_fft=512, hop_length=256, win_length=400, window='hann', n_mfcc=40).T, axis=0)

    return mfc_coefficients

In [ ]:
def get_features(file):  # load an individual soundfile
    waveform, sr = lb.load(file, sr=16000)
    sample_rate = sr

    # compute features of soundfile
    chromagram = feature_chromagram(waveform, sample_rate)
    melspectrogram = feature_melspectrogram(waveform, sample_rate)
    mfc_coefficients = feature_mfcc(waveform, sample_rate)

    feature_matrix = np.array([])
    # use np.hstack to stack our feature arrays horizontally to create a feature matrix
    feature_matrix = np.hstack((chromagram, melspectrogram, mfc_coefficients))

    return feature_matrix


In [ ]:

def get_feature_matrix(files, labels):
    temp_feats = list()
    temp_labels = list()
    for file, label in zip(files, labels):
        features = get_features(file)
        temp_feats.append(features)
        temp_labels.append(label)

    return np.array(temp_feats), np.array(temp_labels)

train_features, train_labels = get_feature_matrix(train_files, train_labels)
test_features, test_labels = get_feature_matrix(test_files, test_labels)

train_features.shape

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

train_df = pd.DataFrame(train_features)
test_df = pd.DataFrame(test_features)


def get_features_statistics(df):
    # Check chromagram feature values

    features_df_chromagram = df.loc[:, :11]
    chroma_min = features_df_chromagram.min().min()
    chroma_max =features_df_chromagram.max().max()

    # stack all features into a single series so we don't get a mean of means or stdev of stdevs
    chroma_mean = features_df_chromagram.stack().mean()

    chroma_stdev = features_df_chromagram.stack().std()
    print(f'Chromagram features: \n'
          f'min: {chroma_min:.3f}, \n'
          f'max: {chroma_max:.3f}, \n'
          f'mean: {chroma_mean:.3f}, \n'
          f'deviation: {chroma_stdev:.3f}')

    # Check mel spectrogram feature values
    features_df_melspectrogram = df.loc[:, 12:139]

    mel_min = features_df_melspectrogram.min().min()
    mel_max = features_df_melspectrogram.max().max()

    # stack all features into a single series so we don't get a mean of means or stdev of stdevs
    mel_mean = features_df_melspectrogram.stack().mean()

    mel_stdev = features_df_melspectrogram.stack().std()
    print(f'\n(Mean Mel Spectrogram vector/feature): \n'
          f'min: {mel_min:.3f}, \n'
          f'max: {mel_max:.3f}, \n'
          f'mean: {mel_mean:.3f}, \n'
          f'deviation: {mel_stdev:.3f}')
    #Check MFCC feature values
    features_df_mfcc=df.loc[:,148:179]

    mfcc_min = features_df_mfcc.min().min()
    mfcc_max = features_df_mfcc.max().max()

    #stack all features into a single series so we don't get a mean of means or stdev of stdevs
    mfcc_mean = features_df_mfcc.stack().mean()

    mfcc_stdev = features_df_mfcc.stack().std()
    print (f'\n MFCC features: \n'
          f'min: {mfcc_min:.3f}, \n'
          f'max: {mfcc_max:.3f}, \n'
          f'mean: {mfcc_mean:.3f}, \n'
          f'deviation: {mfcc_stdev:.3f}')
get_features_statistics(train_df)


In [ ]:
scaler = StandardScaler()
train_features_scaled = scaler.fit_transform(train_features)

scaler = MinMaxScaler()
train_features_minmax = scaler.fit_transform(train_features)

scaler = StandardScaler()
test_features_scaled = scaler.fit_transform(test_features)

scaler = MinMaxScaler()
test_features_minmax = scaler.fit_transform(test_features)


In [ ]:
print("Train Standard Scaling")
print(" ".join(["-"]*20))
train_features_scaled_df = pd.DataFrame(train_features_scaled)
get_features_statistics(train_features_scaled_df)
print("\n\n")

print("Train MinMax Scaling")
print(" ".join(["-"]*20))
train_features_minmax_df = pd.DataFrame(train_features_minmax)
get_features_statistics(train_features_minmax_df)
print("\n\n")

print("Test Standard Scaling")
print(" ".join(["-"]*20))
test_features_scaled_df = pd.DataFrame(test_features_scaled)
get_features_statistics(test_features_scaled_df)
print("\n\n")

print("Test MinMax Scaling")
print(" ".join(["-"]*20))
test_features_minmax_df = pd.DataFrame(test_features_minmax)
get_features_statistics(test_features_minmax_df)
print("\n\n")

Observation of Features and Elemmination

In [ ]:
from sklearn.preprocessing import LabelEncoder

import seaborn as sns

import matplotlib.pyplot as plt

encoder = LabelEncoder()
encoder.fit((train_labels.tolist() + test_labels.tolist()))

encoded_train_labels = encoder.transform(train_labels)
encoded_test_labels = encoder.transform(test_labels)

train_labels

In [ ]:
train_df[180]=encoded_train_labels

In [ ]:
corr= train_df.corr()
class_correlation =list(corr[180])

feature_indexes =list(range(181))
sorted_feature_indexes= [x for _,x in sorted(zip(class_correlation, feature_indexes))]

feature_selection= sorted_feature_indexes[:10] + sorted_feature_indexes [171:]

corr

In [ ]:
temp_train_df = train_df[feature_selection]

In [ ]:
temp_train_df = train_df[feature_selection]
from sklearn.model_selection import train_test_split
corr = temp_train_df.corr()

f, ax = plt.subplots(figsize=(11, 11))

sns.heatmap(corr,
            xticklabels=corr.columns,
            yticklabels=corr.columns,
            ax=ax)

X_train, X_test, y_train, y_test = train_test_split( train_features,encoded_train_labels, test_size=0.3, random_state=76)


Train, Validation and Test

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
###for unscaled test/train set###
X_train, X_test, y_train, y_test = train_test_split( train_features,encoded_train_labels, test_size=0.3, random_state=76)


In [ ]:
####for Standard scaled test/train set####
X_train_scaled, X_test_scaled, _ ,_  = train_test_split( train_features_scaled,encoded_train_labels, test_size=0.3, random_state=76)


In [ ]:
#### MINMAX scaled test/train set####
X_train_minmax, X_test_minmax, _ ,_  = train_test_split( train_features_minmax,encoded_train_labels, test_size=0.3, random_state=76)


Model Selection

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.linear_model import LogisticRegression

In [ ]:
classification_models = [KNeighborsClassifier(),
                          SVC(kernel='linear'),
                          SVC(kernel='rbf'),
                          DecisionTreeClassifier(),
                          RandomForestClassifier(),
                          LogisticRegression()]

In [ ]:
scores = []
for model in classification_models:
    model=model
    model.fit(X_train, y_train)
    score_normal = model.score(X_test, y_test)
    test_normal = model.score(test_features, encoded_test_labels)

    model.fit(X_train_scaled, y_train)
    score_scaled = model.score(X_test_scaled,y_test)
    test_scaled = model.score(test_features_scaled,encoded_test_labels)

    model.fit(X_train_minmax, y_train)
    score_minmax = model.score(X_test_minmax,y_test)
    test_minmax = model.score(test_features_minmax,encoded_test_labels)

    model_name = type(model).__name__
    if model_name=='SVC' and model.kernel=='rbf': model_name+=' RBF kernal'
    scores.append((model_name,(f'{100*score_normal:.2f}%'),(f'{100*score_scaled:.2f}%'),(f'{100*score_minmax:.2f}%'),
                   (f'{100*test_normal:.2f}%'),(f'{100*test_scaled:.2f}%'),(f'{100*test_minmax:.2f}%')))
    scores_df = pd.DataFrame(scores,columns=['Classifier', 'Accuracy Normal', 'Accuracy Scaled', 'Accuracy MinMax', 'Test Normal', 'Test Scaled', 'Test MinMax'])

In [ ]:
scores_df

Model Observation

In [ ]:
import warnings
from sklearn.model_selection import KFold
warnings.filterwarnings('ignore')
kf = KFold(n_splits=10,shuffle=True,random_state=76)
model = LogisticRegression()

In [ ]:
for train_index,test_index in kf.split(train_features):
  X_train,X_test = train_features[train_index],train_features[test_index]
  Y_train,Y_test = encoded_train_labels[train_index],encoded_train_labels[test_index]
  model.fit(X_train,Y_train)
  print(model.score(X_test,Y_test))

Result Observation

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sn

In [ ]:
test_predictions = model.predict(test_features)
test_groundtruth = encoded_test_labels


In [ ]:
conf_matrix = confusion_matrix(test_groundtruth, test_predictions)
conf_matrix_norm = confusion_matrix(test_groundtruth, test_predictions, normalize='true')
class_list = ["twenties","thirties","fourties","fifties"]
class_name = [cls for cls in class_list]
confmatrix_df = pd.DataFrame(conf_matrix, index=class_name, columns=class_name)
confmatrix_df_norm = pd.DataFrame(conf_matrix_norm, index=class_name, columns=class_name)

In [ ]:
plt.figure(figsize=(16,6))
plt.subplot(1,2,1)
plt.title('Confusion Matrix')
sn.heatmap(confmatrix_df, annot=True,fmt='d',annot_kws={"size":18})
plt.subplot(1,2,2)
plt.title('Normalized Confusion Matrix')
sn.heatmap(confmatrix_df_norm, annot=True,annot_kws={"size":13})

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

y_pred = model.predict(test_features)

y_test = encoded_test_labels

print(f'Test Set Accuracy score = {100*accuracy_score(y_test, y_pred):.3f}%')

print(f'Test Set Precision score = {100*precision_score(y_test, y_pred, average="macro"):.3f}%')
print(f'Test Set Recall score = {100*recall_score(y_test, y_pred, average="macro"):.3f}%')

print(f'Test Set F-score score = {100*f1_score(y_test, y_pred, average="macro"):.3}%')

Time Complexity

In [ ]:
start = time.time()
model.fit(train_features,encoded_train_labels)
end = time.time()
print("model train time with  complete features", end-start)

2b. 1D Convolutional Neural Network (CNN) with Concatenated Features: Will a 1D
CNN perform better on a feature set created by concatenating MFCCs, Melspectrograms, and Chroma features (as in Tutorial 3), compared to using them
individually?

Preparation of the data:

In [ ]:
dataset = load_dataset("mozilla-foundation/common_voice_11_0","ta")
train = dataset["train"]
test = dataset["test"]

In [ ]:
def feature_mfcc(waveform,sample_rate):
  mfc_coefficients = np.mean(lb.feature.mfcc(y=waveform,sr=sample_rate, n_fft=512, hop_length=200, win_length=400,window='hann', n_mfcc=40).T, axis=0)
  return mfc_coefficients
def feature_chromagram(waveform,sample_rate):
  chromagram_coefficients = np.mean(lb.feature.chroma_stft(y=waveform,sr=sample_rate, n_fft=512, hop_length=200, win_length=400).T, axis=0)
  return chromagram_coefficients
def feature_melspectrogram(waveform,sample_rate):
  melspectrogram_coefficients = np.mean(lb.feature.melspectrogram(y=waveform,sr=sample_rate, n_mels = 128, n_fft=512, hop_length = 200, fmax=8000, win_length=400, window='hann').T, axis=0)
  return melspectrogram_coefficients



In [ ]:
def get_features(file):

#load an individual soundfile

  waveform, sr = lb.load(file, sr=16000)

  sample_rate = sr

#compute features of soundfile

  chromagram = feature_chromagram(waveform, sample_rate)

  melspectrogram = feature_melspectrogram(waveform, sample_rate)
  mfc_coefficients = feature_mfcc(waveform, sample_rate)

  feature_matrix = np.array([])

#use np.hstack to stack our feature arrays horizontally to create a feature matrix
  feature_matrix = np.hstack((chromagram, melspectrogram, mfc_coefficients))

  return feature_matrix

In [ ]:
def get_feature_matrix(files, labels):

  temp_feats = list()

  temp_labels = list()

  for file, label in zip(files, labels):

    features = get_features(file)

    temp_feats.append(features)
    temp_labels.append(label)

  return np.array(temp_feats), np.array(temp_labels)



In [ ]:
train_features, train_labels = get_feature_matrix(train_files, train_labels)
test_features, test_labels = get_feature_matrix(test_files, test_labels)

In [ ]:
train_df = pd.DataFrame(train_features)
test_df = pd.DataFrame (test_features)

Train, Validation and Test



In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


In [ ]:
encoder = LabelEncoder()
encoder.fit((train_labels.tolist() + test_labels.tolist()))

encoded_train_labels = encoder.transform(train_labels)
encoded_test_labels = encoder.transform(test_labels)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(train_features,encoded_train_labels,test_size=0.2,random_state=69)

test_features = test_features
encoded_test_labels = encoded_test_labels

Neural Network Classifier

In [ ]:
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.preprocessing import LabelEncoder

In [ ]:
#baseline model

def create_baseline():

#create model

  model = Sequential()

  model.add(Dense (60, input_dim=train_features.shape[1], activation='relu'))
  model.add(Dense(1, activation="sigmoid"))

#Compile model

  model.compile(loss = 'binary_crossentropy', optimizer='adam', metrics=["accuracy"])

  return model

In [ ]:
model = create_baseline()

In [ ]:
model.summary()

In [ ]:
model.fit(X_train,y_train,batch_size=5,epochs=76,verbose=1,validation_data=(X_test,y_test))

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

y_pred = model.predict(test_features)

y_pred = np.round(y_pred).astype(int)

y_true = encoded_test_labels

print(f'Test Set Accuracy score = {100*accuracy_score(y_true, y_pred):.3f}%')

print(f'Test Set Precision score print = {100*precision_score(y_true, y_pred, average="macro"):.3f}%')
print(f'Test Set Recall score = {100*recall_score(y_true, y_pred, average="macro"):.3f}%')

print(f'Test Set F-score score = {100*f1_score(y_true, y_pred, average="macro"):.3}%')